# The competitors

The published matchers over the same pairs as the arms, scored by the same
code, and nothing else: no arm is read here. Magneto is the published method
with its LLM reranker, run on our cached Qwen order through OpenRouter. It
costs money, is priced in section 1 and is sent only when `DRY_RUN` is False.

Every run is cached in `results/competitors/`, the LLM ones after each
column, so a second pass is free and an interrupted one resumes. The scores
are written to `results/metrics_competitors_<corpus>.csv`, competitors only.

## 0. What to run

In [5]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from corpora import load
from evaluation import competitors
from evaluation import scoring as report
from schemaoutis import cache

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CORPUS = "gdc"                  # or "valentine/opendata", "valentine/chembl", ...
STUDIES = None                  # None = every pair in the corpus

# Free matchers: any name in `competitors.MATCHERS`, or `magneto__<order>` for
# Magneto on one of our cached orders (`magneto-no-bp__<order>` without its
# bipartite stage). Plain `magneto` is magneto-python's own mpnet model.
MATCHERS = ["coma", "coma-instances", "jaccard-levenshtein"]

# Magneto with its LLM reranker, its shortlist built on our cached order.
# These cost money. OpenRouter ids; price is $ per 1M tokens (in, out).
LLM_RETRIEVERS = ["magneto"]
LLMS = ["deepseek/deepseek-v4.1-flash"]
BP = True                       # Magneto's bipartite stage, part of its method
WORKERS = 8                     # columns sent at once
PRICE = {"deepseek/deepseek-v4.1-flash": (0.14, 0.42)}
# Tokens per column (in, out) used to price a configuration with nothing done
# yet; measured on GDC Clark, where V4.1 Flash reasons before it answers.
TOKENS_PER_COLUMN = (861, 2603)

# True prices the LLM runs and sends nothing. The free matchers always run.
DRY_RUN = False

# Not `results/runs/`, where they would be read as arms. RUNS is only read,
# for the cached retrieval orders Magneto starts from.
RESULTS = REPO / "results" / "competitors"
RUNS = REPO / "results" / "runs"
METRICS = REPO / "results" / f"metrics_competitors_{cache.fs_safe(CORPUS)}.csv"

PAIRS = load(CORPUS, names=STUDIES)
N_SRC = sum(p["source_df"].shape[1] for p in PAIRS)
LLM_TAGS = [competitors.magneto_llm_tag(r, m, bp=BP)
            for r in LLM_RETRIEVERS for m in LLMS]
print(f"corpus     {CORPUS}: {len(PAIRS)} studies, {N_SRC} source columns")
print(f"matchers   {MATCHERS}")
print(f"with LLM   {LLM_TAGS}")
print(f"runs ->    {RESULTS}")
print(f"metrics -> {METRICS}")

corpus     gdc: 10 studies, 569 source columns
matchers   ['coma', 'coma-instances', 'jaccard-levenshtein']
with LLM   ['magneto-llm__magneto__openrouter__deepseek-v4.1-flash']
runs ->    C:\Users\muril\SchemaOutis\results\competitors
metrics -> C:\Users\muril\SchemaOutis\results\metrics_competitors_gdc.csv


## 1. What is left, and what it costs

In [6]:
TODO = [(m, p["name"]) for m in MATCHERS for p in PAIRS
        if competitors.load(RESULTS, CORPUS, p["name"], m) is None]
print(f"free matchers: {len(TODO)} of {len(MATCHERS) * len(PAIRS)} run(s) to do")

PLAN = pd.DataFrame(competitors.magneto_llm_plan(
    PAIRS, results=RESULTS, corpus=CORPUS, runs=RUNS,
    retrievers=LLM_RETRIEVERS, llms=LLMS, bp=BP))
NO_ORDER = PLAN[~PLAN["order_on_disk"]]
if len(NO_ORDER):
    print(f"\n{len(NO_ORDER)} pair(s) have no cached retrieval order, so the "
          "LLM runs cannot build a shortlist for them. Build the order with "
          "run_arms first:")
    display(NO_ORDER[["retriever", "pair"]].drop_duplicates())

COST = PLAN.groupby(["matcher", "llm"], as_index=False)[
    ["columns", "done", "todo", "calls", "tokens_in", "tokens_out"]].sum()
seen = COST["done"] > 0
COST["in_per_col"] = (COST["tokens_in"] / COST["done"]).where(
    seen, TOKENS_PER_COLUMN[0])
COST["out_per_col"] = (COST["tokens_out"] / COST["done"]).where(
    seen, TOKENS_PER_COLUMN[1])
COST["dollars_left"] = [
    r.todo * (r.in_per_col * PRICE[r.llm][0] + r.out_per_col * PRICE[r.llm][1])
    / 1e6 for r in COST.itertuples()]
display(COST.round(2))
print(f"\nLLM: {int(COST['todo'].sum()):,} column(s) left, about "
      f"${COST['dollars_left'].sum():.2f}, one call per column")

free matchers: 0 of 30 run(s) to do


,matcher,llm,columns,done,todo,calls,tokens_in,tokens_out,in_per_col,out_per_col,dollars_left
0,magneto-llm__magneto__openrouter__deepseek-v4....,deepseek/deepseek-v4.1-flash,569,0,569,0,0,0,861.0,2603.0,0.69



LLM: 569 column(s) left, about $0.69, one call per column


## 2. Run

In [7]:
ROWS = pd.DataFrame(competitors.run(PAIRS, MATCHERS, results=RESULTS,
                                    corpus=CORPUS, runs=RUNS))
display(ROWS)

if DRY_RUN:
    print(f"DRY_RUN is True: Magneto with its LLM was NOT run "
          f"({int(COST['todo'].sum()):,} columns left, about "
          f"${COST['dollars_left'].sum():.2f}). Set DRY_RUN = False to send it.")
elif len(NO_ORDER):
    raise RuntimeError("some pairs have no cached retrieval order; see above")
else:
    LLM_ROWS = pd.DataFrame(competitors.run_magneto_llm_all(
        PAIRS, results=RESULTS, corpus=CORPUS, runs=RUNS,
        retrievers=LLM_RETRIEVERS, llms=LLMS, bp=BP, workers=WORKERS))
    display(LLM_ROWS)

cached coma                 Cao          37/39 columns answered      5.4s
cached coma                 Clark        16/16 columns answered      0.9s
cached coma                 Dou          113/179 columns answered      3.7s
cached coma                 Gillette     51/67 columns answered      1.9s
cached coma                 Huang        34/37 columns answered      1.6s
cached coma                 Krug         46/54 columns answered      1.5s
cached coma                 McDermott    21/22 columns answered      1.0s
cached coma                 Satpathy     60/82 columns answered      2.4s
cached coma                 Vasaikar     37/45 columns answered      1.3s
cached coma                 Wang         27/28 columns answered      1.4s
cached coma-instances       Cao          36/39 columns answered      4.2s
cached coma-instances       Clark        16/16 columns answered      2.0s
cached coma-instances       Dou          121/179 columns answered     16.3s
cached coma-instances       Gillet

,corpus,pair,matcher,wall_s,n_source_columns,n_target_columns,n_pairs_proposed,cached,n_answered
0,gdc,Cao,coma,5.40,39,736,113,True,37
1,gdc,Clark,coma,0.86,16,736,33,True,16
2,gdc,Dou,coma,3.66,179,736,268,True,113
3,gdc,Gillette,coma,1.92,67,736,97,True,51
4,gdc,Huang,coma,1.55,37,736,98,True,34
5,gdc,Krug,coma,1.48,54,736,84,True,46
6,gdc,McDermott,coma,0.98,22,736,49,True,21
7,gdc,Satpathy,coma,2.36,82,736,118,True,60
8,gdc,Vasaikar,coma,1.29,45,736,93,True,37
9,gdc,Wang,coma,1.35,28,736,47,True,27


--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Cao
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Clark
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Dou


23:25:48 - LiteLLM:WARNING: core_helpers.py:108 - Unmapped finish_reason 'error', defaulting to 'stop'
23:26:27 - LiteLLM:WARNING: core_helpers.py:108 - Unmapped finish_reason 'error', defaulting to 'stop'


--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Gillette
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Huang
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Krug
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / McDermott
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Satpathy


02:22:17 - LiteLLM:WARNING: core_helpers.py:108 - Unmapped finish_reason 'error', defaulting to 'stop'
C:\Users\muril\AppData\Local\Programs\Python\Python314\Lib\site-packages\magneto\llm_reranker.py:108: UserWarning: Error parsing JSON response: 'NoneType' object is not subscriptable
Raw response: None
  warnings.warn(


--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Vasaikar
--- magneto-llm__magneto__openrouter__deepseek-v4.1-flash / Wang


,matcher,pair,columns,fallback,wall_s
0,magneto-llm__magneto__openrouter__deepseek-v4....,Cao,39,0,545.53
1,magneto-llm__magneto__openrouter__deepseek-v4....,Clark,16,0,27.42
2,magneto-llm__magneto__openrouter__deepseek-v4....,Dou,179,0,1814.90
3,magneto-llm__magneto__openrouter__deepseek-v4....,Gillette,67,0,1736.61
4,magneto-llm__magneto__openrouter__deepseek-v4....,Huang,37,0,2323.06
5,magneto-llm__magneto__openrouter__deepseek-v4....,Krug,54,0,2475.87
6,magneto-llm__magneto__openrouter__deepseek-v4....,McDermott,22,0,2055.92
7,magneto-llm__magneto__openrouter__deepseek-v4....,Satpathy,82,0,2716.22
8,magneto-llm__magneto__openrouter__deepseek-v4....,Vasaikar,45,0,1076.34
9,magneto-llm__magneto__openrouter__deepseek-v4....,Wang,28,0,38.30


## 3. How they did, saved

Pooled over all studies by `report.metrics`, the function the arm rows use,
so the saved file can be set next to them later. A column the matcher proposed
nothing for counts as declined.

In [8]:
SCORED = report.score_competitors(PAIRS, MATCHERS + LLM_TAGS, results=RESULTS,
                                  corpus=CORPUS)
SCORED["whole"] = (SCORED["n_studies"] == len(PAIRS)) & (SCORED["n_columns"] == N_SRC)

QUALITY = ["matcher", "whole", "n_studies", "n_columns", "n_gold_pairs",
           "recall@1", "recall@10", "recall@20", "mrr", "top1", "recall@gt"]
DECIDING = ["matcher", "precision_committed", "recall_committed",
            "f1_committed", "precision_evaluated", "f1_evaluated",
            "n_declined", "n_without_gold", "abstain_precision"]

print("ranking quality")
display(SCORED[[c for c in QUALITY if c in SCORED.columns]].round(3))
print("\nwhat they committed to")
display(SCORED[[c for c in DECIDING if c in SCORED.columns]].round(3))
for r in SCORED[~SCORED["whole"]].itertuples():
    print(f"not whole: {r.matcher}, {r.n_studies}/{len(PAIRS)} studies, "
          f"{r.n_columns}/{N_SRC} columns")

SCORED.to_csv(METRICS, index=False)
print(f"wrote {len(SCORED)} competitor rows to {METRICS}")

ranking quality


,matcher,whole,n_studies,n_columns,n_gold_pairs,recall@1,recall@10,recall@20,mrr,top1,recall@gt
0,coma,True,10,569,259,0.309,0.402,0.402,0.521,0.485,0.301
1,coma-instances,True,10,569,259,0.378,0.517,0.517,0.627,0.594,0.266
2,jaccard-levenshtein,True,10,569,259,0.166,0.351,0.363,0.304,0.261,0.050
3,magneto-llm__magneto__openrouter__deepseek-v4....,True,10,569,259,0.529,0.772,0.772,0.852,0.830,0.610



what they committed to


,matcher,precision_committed,recall_committed,f1_committed,precision_evaluated,f1_evaluated,n_declined,n_without_gold,abstain_precision
0,coma,0.181,0.485,0.264,0.141,0.218,127,404,0.937
1,coma-instances,0.218,0.594,0.319,0.172,0.267,120,404,0.900
2,jaccard-levenshtein,0.116,0.261,0.161,0.076,0.117,199,404,0.839
3,magneto-llm__magneto__openrouter__deepseek-v4....,0.242,0.830,0.374,0.241,0.373,2,404,1.000


wrote 4 competitor rows to C:\Users\muril\SchemaOutis\results\metrics_competitors_gdc.csv
